# Sprint 1 — EDA: Análisis Exploratorio de Datos Agrovoltaicos Dinámicos

**Proyecto:** Análisis y Optimización Operativa de Sistemas Agrovoltaicos Dinámicos  
**Organización:** Sostenibilidad y Ciencia  
**Sprint:** 1 de 4 — EDA e insights  
**Entregable:** E02 — Notebook/script replicable del EDA  
**Autor:** Daniel Álvarez / Equipo Chacorocalfarobar  
**Fecha:** 2026-04-17  

---

## Objetivo del notebook

Este notebook cubre el **Entregable E02** del Sprint 1 y sirve de base para el **E01 (Informe EDA)**. El objetivo no es un EDA genérico, sino un análisis orientado al dominio agrovoltaico dinámico: entender cómo la rotación de paneles afecta simultáneamente a la producción energética y al microclima del cultivo.

**Misión transversal:** los análisis siempre consideran el equilibrio energía–cultivo, nunca solo energía.

### Estructura del notebook
1. Carga e inventario de datasets
2. Auditoría de calidad de datos
3. Normalización temporal inicial
4. Análisis univariante
5. Análisis temporal
6. Correlaciones y relaciones entre variables
7. Análisis orientado al dominio agrovoltaico
8. Hipótesis sobre sensores y espacios de investigación
9. Conclusiones del EDA y próximos pasos

---
## 0. Configuración e imports

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_colwidth', 60)

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ── Rutas ──────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd())           # sprint1/
DATA_DIR     = NOTEBOOK_DIR.parent / 'data'
OUT_DIR      = NOTEBOOK_DIR / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

print(f'DATA_DIR  : {DATA_DIR}')
print(f'OUT_DIR   : {OUT_DIR}')
print(f'pandas    : {pd.__version__}')
print(f'numpy     : {np.__version__}')

---
## 1. Carga e inventario de datasets

Detectamos automáticamente todos los CSV en `data/`, identificamos la variable temporal, describimos brevemente qué información contiene cada fichero e inventariamos su tamaño.

> **Nota técnica:** Los ficheros exportados desde el sistema de adquisición incluyen dos peculiaridades:
> - Primera línea `sep=,` (directiva Excel) que hay que saltar.
> - Los valores numéricos contienen la unidad embebida como string (p. ej. `"28.8 °C"`, `"1024 W/m²"`). Se limpiará en la normalización.
> - Los ficheros de precipitación de alta frecuencia (~600k filas) se tratan con muestreo para no saturar memoria.

In [ ]:
# ── Helpers de carga ────────────────────────────────────────────────────────

def strip_unit(series: pd.Series) -> pd.Series:
    """Extrae el valor numérico de strings tipo '28.8 °C', '1024 W/m²', '41.4 °'."""
    if series.dtype == object:
        cleaned = series.astype(str).str.extract(r'([\-\d\.]+)', expand=False)
        return pd.to_numeric(cleaned, errors='coerce')
    return series


def detect_sep_line(filepath: Path) -> int:
    """Devuelve cuántas filas de cabecera extra saltar (0 o 1 si tiene 'sep=')."""
    with open(filepath, 'r', encoding='utf-8-sig') as f:
        first = f.readline().strip()
    return 1 if first.lower().startswith('sep=') else 0


def load_csv(filepath: Path, nrows: int = None) -> pd.DataFrame:
    """Carga un CSV manejando la línea sep= y el BOM UTF-8."""
    skip = detect_sep_line(filepath)
    df = pd.read_csv(
        filepath,
        skiprows=skip,
        encoding='utf-8-sig',
        nrows=nrows,
        low_memory=False
    )
    # Normalizar la columna temporal
    time_cols = [c for c in df.columns if c.strip().lower() == 'time']
    if time_cols:
        df = df.rename(columns={time_cols[0]: 'Time'})
        df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
    return df


def strip_all_units(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica strip_unit a todas las columnas no-Time."""
    df = df.copy()
    for col in df.columns:
        if col != 'Time':
            df[col] = strip_unit(df[col])
    return df

In [ ]:
# ── Inventario ──────────────────────────────────────────────────────────────

# Etiquetas semánticas por fichero (derivadas del nombre y exploración manual)
SEMANTIC_LABELS = {
    'AIR_AIR Temperatures':            'Temperatura del aire — sensores R1, WS100, S1, S2',
    'AIR_ePAR':                        'ePAR (PAR fotosintético) — sensores R1, S1, S2',
    'Intrensiy_Precipitation':         'Intensidad de precipitación — alta frecuencia (~s)',
    'PV_Irradiance':                   'Irradiancia GPOA y Albedo — secciones S1 y S2',
    'PV_PV panel temperatures-data':   'Temperatura de paneles FV — S1 y S2',
    'PV_PV panel temperatures_ Section':'Temperatura paneles sección S1 (detalle)',
    'Precipitation (Total':            'Precipitación acumulada (cumsum) — alta frecuencia',
    'Precipitation-data-as':           'Intensidad precipitación (duplicado joinbyfield)',
    'Precipitation_quantity':          'Cantidad precipitación acumulada',
    'Precipitation_type_-data-2026-02-12 09':  'Tipo precipitación (WMO wawa)',
    'Precipitation_type_-data-2026-02-12 09_01': 'Tipo precipitación (etiqueta texto)',
    'SOIL_Temperature':                'Temperatura del suelo — sensores R1, S1, S2',
    'SOIL_VWC':                        'VWC (Humedad volumétrica suelo) — R1, S1, S2',
    'Tracking angles':                 'Ángulos de tracking paneles — trackers M01–M10',
    'Wind direction':                  'Dirección del viento (predominante)',
    'Wind speed':                      'Velocidad del viento (km/h)',
}

def get_label(stem):
    for k, v in SEMANTIC_LABELS.items():
        if k in stem:
            return v
    return 'Sin etiqueta'


# Ficheros de precipitación de alta frecuencia (tratar aparte)
HIGH_FREQ_PATTERNS = ['Intrensiy_Precipitation', 'Precipitation']

def is_high_freq(fname):
    return any(p in fname for p in HIGH_FREQ_PATTERNS)


inventory_rows = []
csv_files = sorted(DATA_DIR.glob('*.csv'))

for fp in csv_files:
    size_kb = fp.stat().st_size / 1024
    # Para ficheros grandes solo leemos las primeras 5000 filas para el inventario
    nrows_sample = 5000 if is_high_freq(fp.name) else None
    try:
        df_tmp = load_csv(fp, nrows=nrows_sample)
        nrows_real = int(fp.stat().st_size / (fp.stat().st_size / max(len(df_tmp),1))) if nrows_sample else len(df_tmp)
        has_time = 'Time' in df_tmp.columns
        t_min = df_tmp['Time'].min() if has_time else 'N/A'
        t_max = df_tmp['Time'].max() if has_time else 'N/A'
        n_cols = df_tmp.shape[1]
        # Calcular cobertura real de filas sin contar la línea sep
        with open(fp, 'r', encoding='utf-8-sig') as f:
            real_lines = sum(1 for _ in f) - detect_sep_line(fp) - 1  # -1 header
        inventory_rows.append({
            'Fichero': fp.name[:60],
            'Tamaño (KB)': round(size_kb, 1),
            'Filas': real_lines,
            'Columnas': n_cols,
            'Time_min': str(t_min)[:16],
            'Time_max': str(t_max)[:16],
            'Alta_freq': is_high_freq(fp.name),
            'Descripción': get_label(fp.stem)
        })
    except Exception as e:
        inventory_rows.append({'Fichero': fp.name[:60], 'Error': str(e)})

inv_df = pd.DataFrame(inventory_rows)
display(inv_df.to_string(index=False))
inv_df.to_csv(OUT_DIR / 'inventario_datasets.csv', index=False)
print('\n=> Inventario guardado en outputs/inventario_datasets.csv')

### 1.1 Carga del dataset principal (datasets de trabajo)

Para el análisis principal se trabaja con los **8 datasets de sensores de baja/media frecuencia** (≤6h). Los ficheros de precipitación de alta frecuencia se tratan separadamente con muestreo temporal. Se excluyen duplicados evidentes de precipitación.

In [ ]:
# Mapa de alias cortos para datasets
DATASETS = {
    'air_temp':      'AIR_AIR Temperatures (ALL)-data-as-joinbyfield-2026-02-12 08_56_48.csv',
    'epar':          'AIR_ePAR (all parameters)-data-as-joinbyfield-2026-02-12 09_00_37.csv',
    'irradiance':    'PV_Irradiance-data-as-joinbyfield-2026-02-12 14_17_50.csv',
    'pv_temp':       'PV_PV panel temperatures-data-as-joinbyfield-2026-02-12 14_18_22.csv',
    'soil_temp':     'SOIL_Temperature-data-as-joinbyfield-2026-02-12 08_54_57.csv',
    'soil_vwc':      'SOIL_VWC-data-as-joinbyfield-2026-02-12 08_57_18.csv',
    'tracking':      'Tracking angles-data-as-joinbyfield-2026-02-12 14_24_26.csv',
    'wind_speed':    'Wind speed-data-2026-02-12 09_05_40.csv',
    'wind_dir':      'Wind direction-data-2026-02-12 14_25_08.csv',
    'precip_type':   'Precipitation_type_-data-2026-02-12 09_01_21.csv',
    'precip_cumsum': 'Precipitation_quantity_Precipitation (Total  Difference cumulative sum)-data-2026-02-12 09_05_15.csv',
}

dfs = {}
for key, fname in DATASETS.items():
    fp = DATA_DIR / fname
    if not fp.exists():
        print(f'[WARN] No encontrado: {fname}')
        continue
    # Precipitación alta frecuencia: muestrear cada 10 filas para exploración
    if key in ('precip_type', 'precip_cumsum'):
        df_raw = load_csv(fp)
        dfs[key] = df_raw.iloc[::10].reset_index(drop=True)
    else:
        dfs[key] = load_csv(fp)
    print(f'[OK] {key:15s} → {dfs[key].shape[0]:>5} filas x {dfs[key].shape[1]} cols')

print('\nDatasets cargados:', list(dfs.keys()))

---
## 2. Auditoría de calidad de datos

Para cada dataset evaluamos: tipos de variable, nulos, duplicados, rangos anómalos, variables constantes, formato de timestamps y posibles columnas problemáticas.

In [ ]:
def quality_report(name: str, df: pd.DataFrame) -> dict:
    """Genera métricas de calidad para un dataframe."""
    n = len(df)
    non_time_cols = [c for c in df.columns if c != 'Time']
    
    # Strips units para evaluar rangos
    df_num = strip_all_units(df[non_time_cols]) if non_time_cols else pd.DataFrame()
    
    null_pct_mean  = df_num.isnull().mean().mean() * 100 if not df_num.empty else 0
    dup_rows       = df.duplicated().sum()
    time_nulls     = df['Time'].isnull().sum() if 'Time' in df.columns else 0
    const_cols     = [c for c in df_num.columns if df_num[c].nunique() <= 1]
    high_null_cols = [c for c in df_num.columns if df_num[c].isnull().mean() > 0.95]
    
    # Frecuencia temporal aproximada
    freq_str = 'N/A'
    if 'Time' in df.columns and n > 2:
        deltas = df['Time'].dropna().sort_values().diff().dropna()
        if not deltas.empty:
            median_delta = deltas.median()
            freq_str = str(median_delta)
    
    return {
        'Dataset':           name,
        'Filas':             n,
        'Columnas':          len(df.columns),
        'Nulos promedio %':  round(null_pct_mean, 1),
        'Filas duplicadas':  int(dup_rows),
        'Time nulos':        int(time_nulls),
        'Cols constantes':   len(const_cols),
        'Cols >95% nulo':    len(high_null_cols),
        'Frecuencia aprox.': freq_str,
        'Cols const. nombres': ', '.join(const_cols[:5]) or '-',
    }


quality_rows = []
for name, df in dfs.items():
    quality_rows.append(quality_report(name, df))

quality_df = pd.DataFrame(quality_rows)
display(quality_df[[
    'Dataset','Filas','Columnas','Nulos promedio %',
    'Filas duplicadas','Time nulos','Cols constantes',
    'Cols >95% nulo','Frecuencia aprox.'
]])
quality_df.to_csv(OUT_DIR / 'quality_report.csv', index=False)
print('\n=> Guardado en outputs/quality_report.csv')

In [ ]:
# ── Detalle de nulos por columna para datasets principales ──────────────────

MAIN_DATASETS = ['air_temp', 'epar', 'irradiance', 'pv_temp', 'soil_temp', 'soil_vwc', 'tracking']

fig, axes = plt.subplots(len(MAIN_DATASETS), 1, figsize=(14, 2.2 * len(MAIN_DATASETS)))
fig.suptitle('Porcentaje de valores nulos por columna', fontsize=13, fontweight='bold', y=1.01)

for ax, name in zip(axes, MAIN_DATASETS):
    df_n = strip_all_units(dfs[name].drop(columns=['Time'], errors='ignore'))
    null_pct = df_n.isnull().mean() * 100
    colors = ['#e74c3c' if v > 80 else '#f39c12' if v > 30 else '#2ecc71' for v in null_pct]
    null_pct.plot(kind='bar', ax=ax, color=colors, width=0.8)
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_ylabel('%')
    ax.set_ylim(0, 105)
    ax.axhline(80, color='red', linestyle='--', alpha=0.4, label='80%')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUT_DIR / 'nulls_by_column.png', dpi=120, bbox_inches='tight')
plt.show()
print('=> Guardado en outputs/nulls_by_column.png')

### 2.1 Observaciones de calidad

> - **Zona R1** (sensores `R1d40`, `R1d41`, etc.) presenta nulos cercanos al 100% en la mayoría de datasets. Es posible que sea una zona de referencia no instrumentada completamente o con adquisición desactivada en la mayor parte del período.
> - Los valores válidos empiezan predominantemente a partir de **junio 2025**, aunque los timestamps más antiguos datan de febrero 2025.
> - Los trackers M02, M06, M10 muestran valores fijos en torno a **50.6°** en múltiples registros, lo que sugiere posición de stow (aparcamiento) o avería de actuador.
> - Las unidades embebidas en strings (`°C`, `W/m²`, `%`, `km/h`, `umol m-2 s-1`) requieren limpieza antes de cualquier análisis numérico.

---
## 3. Normalización temporal inicial

Analizamos la frecuencia de muestreo real de cada dataset, detectamos huecos temporales y evaluamos la posible desalineación entre datasets.

In [ ]:
def temporal_summary(name: str, df: pd.DataFrame) -> dict:
    if 'Time' not in df.columns:
        return {'Dataset': name, 'Error': 'Sin columna Time'}
    ts = df['Time'].dropna().sort_values()
    if len(ts) < 2:
        return {'Dataset': name, 'Error': 'Menos de 2 timestamps válidos'}
    deltas = ts.diff().dropna()
    median_d = deltas.median()
    max_gap  = deltas.max()
    n_gaps   = (deltas > median_d * 3).sum()  # huecos > 3x frecuencia normal
    return {
        'Dataset':             name,
        'T_inicio':            str(ts.min())[:16],
        'T_fin':               str(ts.max())[:16],
        'Días cubiertos':      round((ts.max() - ts.min()).days, 1),
        'Frecuencia mediana':  str(median_d),
        'Hueco máximo':        str(max_gap),
        'N° huecos grandes':   int(n_gaps),
    }

temporal_rows = [temporal_summary(n, d) for n, d in dfs.items()]
temp_df = pd.DataFrame(temporal_rows)
display(temp_df)
temp_df.to_csv(OUT_DIR / 'temporal_summary.csv', index=False)

In [ ]:
# ── Visualización de cobertura temporal por dataset ─────────────────────────

fig, ax = plt.subplots(figsize=(14, 5))

colors_map = plt.cm.tab10.colors
y_pos = 0
yticks, ylabels = [], []

for i, (name, df) in enumerate(dfs.items()):
    if 'Time' not in df.columns:
        continue
    ts = df['Time'].dropna().sort_values()
    if len(ts) < 2:
        continue
    ax.barh(y_pos, left=ts.min(), width=(ts.max()-ts.min()),
            height=0.6, color=colors_map[i % 10], alpha=0.7, label=name)
    yticks.append(y_pos)
    ylabels.append(name)
    y_pos += 1

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
ax.set_title('Cobertura temporal por dataset', fontsize=13, fontweight='bold')
ax.set_xlabel('Fecha')
plt.tight_layout()
plt.savefig(OUT_DIR / 'temporal_coverage.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Detección de huecos mayores de 24h en datasets de sensores ──────────────

print('=== Huecos > 24h por dataset ===')
for name in MAIN_DATASETS:
    df = dfs[name]
    ts = df['Time'].dropna().sort_values()
    gaps = ts.diff().dropna()
    big_gaps = gaps[gaps > pd.Timedelta('24h')]
    if len(big_gaps) > 0:
        print(f'\n{name} — {len(big_gaps)} hueco(s) > 24h:')
        for idx, g in big_gaps.items():
            t_antes = ts.loc[idx - 1] if idx > 0 else 'N/A'
            print(f'   {t_antes}  →  duración hueco: {g}')
    else:
        print(f'{name} — sin huecos > 24h')

### 3.1 Observaciones temporales

> - La mayoría de datasets de sensores presenta **frecuencia de 6 horas** (tracking, soil, air_temp). El dataset de irradiancia tiene **resolución de 2 horas**.
> - Los ficheros de precipitación son de alta frecuencia (~segundos), incompatibles para un merge directo sin resampleo.
> - El período con **datos reales** (no nulos) comienza aproximadamente el **18 de junio de 2025**. Los meses anteriores contienen timestamps pero valores ausentes.
> - Existe un posible **hueco de 4 meses** (febrero–mayo 2025) en varios datasets. Esto puede responder a una fase de instalación o puesta en marcha de los sensores.
> - Para el Sprint 2 (modelización) será necesario hacer **resampleo temporal común** antes de unir los datasets.

---
## 4. Análisis univariante

Analizamos las variables numéricas más relevantes para el proyecto: ángulo de tracking, irradiancia, temperatura de panel, temperatura del aire, temperatura del suelo, VWC y ePAR.

In [ ]:
# ── Preparar versiones numéricas de los datasets principales ────────────────

dfs_num = {k: strip_all_units(dfs[k]) for k in MAIN_DATASETS}
# Restaurar Time
for k in MAIN_DATASETS:
    dfs_num[k]['Time'] = dfs[k]['Time']

print('Versiones numéricas preparadas.')

In [ ]:
def univariate_stats(df_num: pd.DataFrame, exclude_cols=('Time',)) -> pd.DataFrame:
    cols = [c for c in df_num.select_dtypes(include=np.number).columns
            if c not in exclude_cols]
    stats = df_num[cols].describe().T
    stats['null%'] = df_num[cols].isnull().mean().values * 100
    stats['skew']  = df_num[cols].skew().values
    stats['IQR']   = (stats['75%'] - stats['25%'])
    return stats.round(3)


for name in ['tracking', 'irradiance', 'pv_temp', 'air_temp', 'soil_temp', 'soil_vwc', 'epar']:
    print(f'\n{'='*60}')
    print(f' {name.upper()}')
    print('='*60)
    stats = univariate_stats(dfs_num[name])
    display(stats[['count','mean','std','min','25%','50%','75%','max','null%','skew']].head(20))

In [ ]:
# ── Distribuciones de variables clave ───────────────────────────────────────

KEY_VARS = {
    'tracking':   ['tracker_M01 (actual)', 'tracker_M03 (actual)', 'tracker_M05 (actual)'],
    'irradiance': ['S1d12_Z8AI.pyra.GPOA.Wm2', 'S1d12_Z8AI.pyra.ALBEDO.Wm2',
                   'S2d30_Z8AI.pyra.GPOA.Wm2', 'S2d30_Z8AI.pyra.ALBEDO.Wm2'],
    'air_temp':   ['WS100.Air.TempAvg.degC', 'S1d10_Z8AI.air.T__center.degC',
                   'S2d30_Z8AI.air.T__east.degC'],
    'soil_temp':  ['S1d13_HD3910', 'S1d14_HD3910', 'S2d32_HD3910'],
    'soil_vwc':   ['S1d13_HD3910', 'S1d14_HD3910', 'S2d32_HD3910'],
    'epar':       ['S1d19_SQ618 cal_out', 'S1d20_SQ618 cal_out',
                   'S2d36_SQ618 cal_out', 'S2d37_SQ618 cal_out'],
}

for dsname, cols in KEY_VARS.items():
    df_plot = dfs_num[dsname]
    valid_cols = [c for c in cols if c in df_plot.columns]
    if not valid_cols:
        continue
    n = len(valid_cols)
    fig, axes = plt.subplots(1, n, figsize=(4.5*n, 3.5))
    if n == 1:
        axes = [axes]
    fig.suptitle(f'{dsname.upper()} — Distribuciones', fontweight='bold')
    for ax, col in zip(axes, valid_cols):
        data = df_plot[col].dropna()
        ax.hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
        ax.axvline(data.mean(), color='red', linestyle='--', label=f'μ={data.mean():.1f}')
        ax.axvline(data.median(), color='orange', linestyle=':', label=f'med={data.median():.1f}')
        ax.set_title(col.split('.')[-1][:30], fontsize=9)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'dist_{dsname}.png', dpi=110, bbox_inches='tight')
    plt.show()

print('=> Distribuciones guardadas en outputs/')

In [ ]:
# ── Detección de outliers con método IQR ────────────────────────────────────

def count_outliers_iqr(series: pd.Series, factor: float = 3.0) -> int:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return int(((series < q1 - factor*iqr) | (series > q3 + factor*iqr)).sum())

outlier_rows = []
for dsname, cols in KEY_VARS.items():
    df_plot = dfs_num[dsname]
    for col in cols:
        if col not in df_plot.columns:
            continue
        s = df_plot[col].dropna()
        if len(s) < 10:
            continue
        n_out = count_outliers_iqr(s)
        outlier_rows.append({
            'Dataset': dsname, 'Variable': col,
            'N válidos': len(s), 'Outliers (IQR×3)': n_out,
            'Outlier %': round(n_out/len(s)*100, 1),
            'Min': round(s.min(), 2), 'Max': round(s.max(), 2),
            'Media': round(s.mean(), 2)
        })

out_df = pd.DataFrame(outlier_rows)
display(out_df.sort_values('Outlier %', ascending=False))
out_df.to_csv(OUT_DIR / 'outliers_summary.csv', index=False)

### 4.1 Observaciones univariantes

> - **Ángulo de tracking:** varios trackers presentan un pico marcado en torno a **50.6°** que se repite con alta frecuencia. Puede tratarse de la posición de stow (aparcamiento nocturno o por viento) o de un defecto de actuador. Requiere verificación con el equipo técnico.
> - **GPOA (irradiancia en plano de panel):** valores en rango esperado (0–1100 W/m²) con distribución bimodal (0 = noche/nublado, pico diurno ~700-1000 W/m²).
> - **ePAR:** los valores exteriores (R1 y expuestos directamente) superan los 1300 µmol·m⁻²·s⁻¹, mientras los sensores bajo el dosel muestran 400–900 µmol·m⁻²·s⁻¹, lo que evidencia el **efecto de sombreo de los paneles**.
> - **VWC:** los sensores S1 muestran mayor variabilidad (17–36%) que S2 (21–36%), posible indicador de diferencias en tipo de suelo o manejo.
> - **Temperatura suelo:** rango 25–32°C en verano, con diferencias de ~2°C entre posiciones. Plausible físicamente.

---
## 5. Análisis temporal

Analizamos la evolución temporal de las variables clave, identificando tendencias, patrones diarios e intradiarios, y posibles cambios de régimen.

In [ ]:
# ── Evolución temporal variables energéticas ────────────────────────────────

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Evolución temporal — Variables energéticas', fontweight='bold', fontsize=13)

# 1) Ángulo de tracking (todos los trackers)
df_t = dfs_num['tracking'].set_index('Time').sort_index()
track_cols = [c for c in df_t.columns if 'tracker' in c.lower()]
for col in track_cols:
    axes[0].plot(df_t.index, df_t[col], alpha=0.5, linewidth=0.8, label=col[-4:])
axes[0].set_ylabel('Ángulo (°)')
axes[0].set_title('Ángulo de tracking (M01–M10)')
axes[0].legend(ncol=5, fontsize=7)

# 2) Irradiancia GPOA S1 y S2
df_i = dfs_num['irradiance'].set_index('Time').sort_index()
gpoa_cols = [c for c in df_i.columns if 'GPOA' in c]
for col in gpoa_cols:
    axes[1].plot(df_i.index, df_i[col], alpha=0.7, linewidth=0.8, label=col.split('.')[0])
axes[1].set_ylabel('W/m²')
axes[1].set_title('Irradiancia GPOA')
axes[1].legend(fontsize=8)

# 3) Temperatura panel FV
df_pv = dfs_num['pv_temp'].set_index('Time').sort_index()
pv_cols = [c for c in df_pv.columns if c != 'Time']
for col in pv_cols:
    axes[2].plot(df_pv.index, df_pv[col], alpha=0.6, linewidth=0.8, label=col)
axes[2].set_ylabel('°C')
axes[2].set_title('Temperatura de panel FV')
axes[2].legend(ncol=4, fontsize=7)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator())
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUT_DIR / 'temporal_energy_vars.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evolución temporal variables agroclimáticas ─────────────────────────────

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle('Evolución temporal — Variables microclimáticas', fontweight='bold', fontsize=13)

# 1) Temperatura aire (sensores S1 + WS100)
df_a = dfs_num['air_temp'].set_index('Time').sort_index()
air_cols = [c for c in df_a.columns if 'S1' in c or 'S2' in c or 'WS100.Air.TempAvg' in c]
for col in air_cols[:6]:
    axes[0].plot(df_a.index, df_a[col], alpha=0.6, linewidth=0.8, label=col[:20])
axes[0].set_ylabel('°C')
axes[0].set_title('Temperatura del aire')
axes[0].legend(ncol=3, fontsize=7)

# 2) Temperatura suelo
df_st = dfs_num['soil_temp'].set_index('Time').sort_index()
st_cols = [c for c in df_st.columns if 'S1' in c or 'S2' in c]
for col in st_cols[:6]:
    axes[1].plot(df_st.index, df_st[col], alpha=0.6, linewidth=0.8, label=col[:15])
axes[1].set_ylabel('°C')
axes[1].set_title('Temperatura del suelo')
axes[1].legend(ncol=3, fontsize=7)

# 3) VWC
df_vwc = dfs_num['soil_vwc'].set_index('Time').sort_index()
vwc_cols = [c for c in df_vwc.columns if 'S1' in c or 'S2' in c]
for col in vwc_cols[:6]:
    axes[2].plot(df_vwc.index, df_vwc[col], alpha=0.6, linewidth=0.8, label=col[:15])
axes[2].set_ylabel('%')
axes[2].set_title('VWC (humedad volumétrica suelo)')
axes[2].legend(ncol=3, fontsize=7)

# 4) ePAR seleccionado
df_ep = dfs_num['epar'].set_index('Time').sort_index()
epar_sel = [c for c in df_ep.columns if 'cal_out' in c and '16bit' not in c][:6]
for col in epar_sel:
    axes[3].plot(df_ep.index, df_ep[col], alpha=0.6, linewidth=0.8, label=col[:20])
axes[3].set_ylabel('µmol/m²/s')
axes[3].set_title('ePAR fotosintético')
axes[3].legend(ncol=3, fontsize=7)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator())
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUT_DIR / 'temporal_micro_vars.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Patrón intradiario (hora del día) ───────────────────────────────────────
# Usamos los pocos días con datos completos para identificar perfiles diarios.

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Perfil intradiario (mediana por hora)', fontweight='bold', fontsize=13)

plot_configs = [
    ('irradiance', 'S1d12_Z8AI.pyra.GPOA.Wm2', 'GPOA S1 (W/m²)', axes[0,0]),
    ('tracking',   'tracker_M01 (actual)',        'Tracking M01 (°)', axes[0,1]),
    ('epar',       'S1d19_SQ618 cal_out',          'ePAR S1d19 (µmol/m²/s)', axes[1,0]),
    ('soil_temp',  'S1d13_HD3910',                 'T suelo S1d13 (°C)', axes[1,1]),
]

for dsname, col, label, ax in plot_configs:
    df_p = dfs_num[dsname].copy()
    if col not in df_p.columns:
        ax.set_title(f'{label} — columna no disponible')
        continue
    df_p['hour'] = df_p['Time'].dt.hour
    profile = df_p.groupby('hour')[col].agg(['median','quantile']).reset_index() \
                  if False else df_p.groupby('hour')[col].median()
    q25 = df_p.groupby('hour')[col].quantile(0.25)
    q75 = df_p.groupby('hour')[col].quantile(0.75)
    ax.plot(profile.index, profile.values, color='steelblue', linewidth=2, label='mediana')
    ax.fill_between(q25.index, q25.values, q75.values, alpha=0.2, color='steelblue', label='IQR')
    ax.set_title(label)
    ax.set_xlabel('Hora del día')
    ax.set_xticks(range(0, 24, 2))
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'intraday_profiles.png', dpi=120, bbox_inches='tight')
plt.show()

### 5.1 Observaciones temporales clave

> - **Perfil de tracking:** el ángulo sigue claramente el recorrido solar durante el día (valores negativos por la mañana, positivos por la tarde) con transiciones abruptas a la posición de stow. Algunos trackers (M02, M06, M10) quedan fijos —a validar mecánicamente.
> - **Perfil GPOA intradiario:** curva campaniforme diaria esperada con máximo solar al mediodía. La variabilidad entre días sugiere nubosidad variable.
> - **ePAR bajo el dosel:** el máximo diario no coincide exactamente con el máximo de GPOA, lo que puede indicar sombreo asimétrico por la posición del panel. Este desfase temporal es relevante para el modelo de rotación.
> - **VWC:** varía lentamente (escala de días), con pulsos de subida asociados a precipitación. Este comportamiento es coherente con la respuesta hidráulica del suelo.

---
## 6. Correlaciones y relaciones entre variables

Construimos un dataset integrado a resolución 6h (frecuencia común) para analizar correlaciones cruzadas entre todas las variables relevantes.

In [ ]:
# ── Construcción del dataset integrado a 6h ─────────────────────────────────

def resample_to_6h(df_num: pd.DataFrame, agg: str = 'mean') -> pd.DataFrame:
    """Resamplea a 6h sobre el índice Time."""
    df = df_num.copy()
    df = df.set_index('Time').sort_index()
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    return df[num_cols].resample('6h').mean()


# Seleccionar las columnas más representativas de cada dataset
SELECTED_COLS = {
    'tracking':   {
        'tracker_M01 (actual)': 'track_M01',
        'tracker_M03 (actual)': 'track_M03',
        'tracker_M05 (actual)': 'track_M05',
        'tracker_M07 (actual)': 'track_M07',
        'tracker_M09 (actual)': 'track_M09',
    },
    'irradiance': {
        'S1d12_Z8AI.pyra.GPOA.Wm2':   'GPOA_S1',
        'S1d12_Z8AI.pyra.ALBEDO.Wm2':  'Albedo_S1',
        'S2d30_Z8AI.pyra.GPOA.Wm2':   'GPOA_S2',
        'S2d30_Z8AI.pyra.ALBEDO.Wm2':  'Albedo_S2',
    },
    'pv_temp':    {
        'S1 T1': 'Tpanel_S1T1', 'S1 T2': 'Tpanel_S1T2',
        'S2 T1': 'Tpanel_S2T1', 'S2 T2': 'Tpanel_S2T2',
    },
    'air_temp':   {
        'WS100.Air.TempAvg.degC':         'Tair_WS',
        'S1d10_Z8AI.air.T__center.degC':  'Tair_S1_center',
        'S1d10_Z8AI.air.T__east.degC':    'Tair_S1_east',
        'S2d30_Z8AI.air.T__east.degC':    'Tair_S2_east',
    },
    'soil_temp':  {
        'S1d13_HD3910': 'Tsoil_S1d13', 'S1d14_HD3910': 'Tsoil_S1d14',
        'S2d32_HD3910': 'Tsoil_S2d32', 'S2d33_HD3910': 'Tsoil_S2d33',
    },
    'soil_vwc':   {
        'S1d13_HD3910': 'VWC_S1d13', 'S1d14_HD3910': 'VWC_S1d14',
        'S2d32_HD3910': 'VWC_S2d32', 'S2d33_HD3910': 'VWC_S2d33',
    },
    'epar':       {
        'S1d19_SQ618 cal_out': 'ePAR_S1d19',
        'S1d20_SQ618 cal_out': 'ePAR_S1d20',
        'S2d36_SQ618 cal_out': 'ePAR_S2d36',
        'S2d37_SQ618 cal_out': 'ePAR_S2d37',
    },
}

merged_parts = []
for dsname, col_map in SELECTED_COLS.items():
    df_r = resample_to_6h(dfs_num[dsname])
    valid = {k: v for k, v in col_map.items() if k in df_r.columns}
    if valid:
        part = df_r[list(valid.keys())].rename(columns=valid)
        merged_parts.append(part)

df_merged = pd.concat(merged_parts, axis=1)
print(f'Dataset integrado: {df_merged.shape[0]} timesteps x {df_merged.shape[1]} variables')
print(f'Período: {df_merged.index.min()} → {df_merged.index.max()}')
print(f'Nulos promedio: {df_merged.isnull().mean().mean()*100:.1f}%')
df_merged.to_csv(OUT_DIR / 'dataset_integrado_6h.csv')
print('=> Guardado en outputs/dataset_integrado_6h.csv')

In [ ]:
# ── Matriz de correlaciones ─────────────────────────────────────────────────

df_corr = df_merged.dropna(thresh=int(len(df_merged)*0.5), axis=1)  # mínimo 50% datos
corr_matrix = df_corr.corr(method='spearman')  # Spearman para no-linealidades

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', vmin=-1, vmax=1, center=0,
    linewidths=0.5, annot_kws={'size': 7}, ax=ax
)
ax.set_title('Matriz de correlaciones de Spearman — Dataset integrado 6h',
             fontweight='bold', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('=> Guardado en outputs/correlation_matrix.png')

In [ ]:
# ── Correlaciones más fuertes con variable de tracking ──────────────────────

track_cols_merged = [c for c in df_corr.columns if c.startswith('track_')]
if track_cols_merged:
    track_ref = track_cols_merged[0]
    corr_with_track = corr_matrix[track_ref].drop(track_cols_merged).sort_values(key=abs, ascending=False)
    print(f'Correlaciones Spearman con {track_ref}:\n')
    display(corr_with_track.to_frame('Spearman r').head(15))
    
    # Top correlaciones (positivas y negativas)
    fig, ax = plt.subplots(figsize=(10, 5))
    top = corr_with_track.head(15)
    colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in top]
    top.plot(kind='barh', ax=ax, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Correlaciones Spearman con {track_ref}', fontweight='bold')
    ax.set_xlabel('Spearman r')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'correlations_vs_tracking.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Scatter plots clave: tracking vs. variables relevantes ──────────────────

scatter_pairs = [
    ('track_M01', 'GPOA_S1',        'Tracking vs. GPOA S1'),
    ('track_M01', 'ePAR_S1d19',     'Tracking vs. ePAR S1'),
    ('track_M01', 'Tpanel_S1T1',    'Tracking vs. T panel S1'),
    ('GPOA_S1',   'ePAR_S1d19',     'GPOA S1 vs. ePAR S1'),
    ('GPOA_S1',   'Tair_S1_center', 'GPOA S1 vs. T aire S1'),
    ('GPOA_S1',   'Tsoil_S1d13',    'GPOA S1 vs. T suelo S1'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Relaciones bivariadas clave', fontweight='bold', fontsize=13)

for ax, (x_col, y_col, title) in zip(axes.flat, scatter_pairs):
    if x_col not in df_corr.columns or y_col not in df_corr.columns:
        ax.set_title(f'{title}\n(datos no disponibles)')
        continue
    subset = df_corr[[x_col, y_col]].dropna()
    if len(subset) < 5:
        ax.set_title(f'{title}\n(insuf. datos)')
        continue
    r = subset.corr(method='spearman').iloc[0,1]
    ax.scatter(subset[x_col], subset[y_col], alpha=0.4, s=15, color='steelblue')
    ax.set_xlabel(x_col, fontsize=8)
    ax.set_ylabel(y_col, fontsize=8)
    ax.set_title(f'{title}\nSpearman r={r:.2f}', fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'scatter_key_pairs.png', dpi=120, bbox_inches='tight')
plt.show()

### 6.1 Observaciones de correlaciones

> - **Tracking ↔ GPOA:** correlación positiva esperada (más ángulo hacia el sol → más irradiancia en plano del panel). La nube de puntos puede revelar saturación a ángulos extremos.
> - **GPOA ↔ ePAR:** correlación alta, pero el ePAR bajo el dosel es sistemáticamente más bajo que el GPOA expuesto. La diferencia es la "sombra agronómica útil" del sistema.
> - **GPOA ↔ Temperatura suelo/aire:** correlación positiva, con probable retardo temporal (el suelo responde más lento que el aire).
> - **Tracking ↔ ePAR:** relación compleja. Ángulos intermedios pueden maximizar la combinación energía+PAR. Este es el núcleo del problema de optimización del Sprint 2.
> - **Se usa Spearman** (no Pearson) porque las relaciones tracking–microclima son plausiblemente no lineales (efectos de umbral, sombreo asimétrico).

---
## 7. Análisis orientado al dominio agrovoltaico

Este bloque es el más específico del proyecto. Buscamos evidencias de los mecanismos clave del sistema: cómo la rotación de paneles afecta simultáneamente a la producción energética y al microclima del cultivo.

In [ ]:
# ── 7.1 Segmentación por régimen de tracking ────────────────────────────────
# Clasificamos cada timestep en: STOW (~50.6°), TRACKING_ACTIVO, POSICIÓN_FIJA

if 'track_M01' in df_corr.columns:
    df_domain = df_corr.copy()
    angle = df_domain['track_M01']
    
    # Definición de regímenes (a validar con equipo técnico)
    STOW_ANGLE = 50.6
    STOW_TOL   = 0.5
    
    def classify_regime(a):
        if pd.isna(a):
            return 'DESCONOCIDO'
        if abs(a - STOW_ANGLE) < STOW_TOL:
            return 'STOW'
        elif abs(a) < 5:
            return 'HORIZONTAL'
        elif a < 0:
            return 'TRACKING_AM'   # mañana
        else:
            return 'TRACKING_PM'   # tarde
    
    df_domain['regime'] = angle.apply(classify_regime)
    
    print('Distribución de regímenes de tracking:')
    display(df_domain['regime'].value_counts().to_frame('N timesteps'))
else:
    df_domain = df_corr.copy()
    df_domain['regime'] = 'SIN_TRACKING'
    print('[WARN] track_M01 no disponible, análisis de régimen limitado')

In [ ]:
# ── 7.2 Comparativa de variables por régimen de tracking ────────────────────

compare_vars = [
    ('GPOA_S1',        'GPOA S1 (W/m²)'),
    ('ePAR_S1d19',     'ePAR S1 (µmol/m²/s)'),
    ('Tair_S1_center', 'T aire S1 centro (°C)'),
    ('Tsoil_S1d13',    'T suelo S1 (°C)'),
    ('VWC_S1d13',      'VWC S1 (%)'),
]

valid_cvars = [(v, lbl) for v, lbl in compare_vars if v in df_domain.columns]

if valid_cvars:
    fig, axes = plt.subplots(1, len(valid_cvars), figsize=(4.5*len(valid_cvars), 5))
    if len(valid_cvars) == 1:
        axes = [axes]
    fig.suptitle('Distribución de variables por régimen de tracking', fontweight='bold', fontsize=13)
    
    regime_order = ['TRACKING_AM', 'HORIZONTAL', 'TRACKING_PM', 'STOW', 'DESCONOCIDO']
    palette = {'TRACKING_AM': '#3498db', 'HORIZONTAL': '#2ecc71',
               'TRACKING_PM': '#e67e22', 'STOW': '#95a5a6', 'DESCONOCIDO': '#bdc3c7'}
    
    for ax, (var, label) in zip(axes, valid_cvars):
        present_regimes = [r for r in regime_order if r in df_domain['regime'].unique()]
        df_domain.boxplot(column=var, by='regime', ax=ax,
                          positions=range(len(present_regimes)),
                          widths=0.5)
        ax.set_xticklabels(present_regimes, rotation=30, fontsize=8)
        ax.set_title(label, fontsize=9)
        ax.set_xlabel('')
    
    plt.suptitle('Distribución de variables por régimen de tracking', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'vars_by_tracking_regime.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 7.3 Diferencias S1 vs S2 (efecto sección experimental) ─────────────────

section_compare = [
    ('GPOA_S1',     'GPOA_S2',     'GPOA (W/m²)'),
    ('ePAR_S1d19',  'ePAR_S2d36',  'ePAR (µmol/m²/s)'),
    ('Tsoil_S1d13', 'Tsoil_S2d32', 'T suelo (°C)'),
    ('VWC_S1d13',   'VWC_S2d32',   'VWC (%)'),
]

valid_sc = [(a, b, lbl) for a, b, lbl in section_compare
            if a in df_domain.columns and b in df_domain.columns]

if valid_sc:
    fig, axes = plt.subplots(1, len(valid_sc), figsize=(4.5*len(valid_sc), 4))
    if len(valid_sc) == 1:
        axes = [axes]
    fig.suptitle('Comparativa sección S1 vs S2', fontweight='bold', fontsize=13)
    
    for ax, (s1_col, s2_col, label) in zip(axes, valid_sc):
        s1_data = df_domain[s1_col].dropna()
        s2_data = df_domain[s2_col].dropna()
        ax.boxplot([s1_data, s2_data], labels=['S1', 'S2'],
                   patch_artist=True,
                   boxprops=dict(facecolor='#3498db', alpha=0.6))
        ax.set_title(label)
        s1_m, s2_m = s1_data.median(), s2_data.median()
        diff = s2_m - s1_m
        ax.set_xlabel(f'Δmediana S2-S1 = {diff:.2f}', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'section_S1_vs_S2.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 7.4 Análisis de lag: ePAR retardado respecto a GPOA ─────────────────────
# ¿Cuánto tiempo tarda el microclima en responder a cambios de irradiancia?

from pandas.plotting import lag_plot

lag_pairs = [
    ('GPOA_S1', 'Tsoil_S1d13', 'GPOA → T suelo S1'),
    ('GPOA_S1', 'VWC_S1d13',   'GPOA → VWC S1'),
]

lag_results = []
fig, axes = plt.subplots(1, len(lag_pairs)*4, figsize=(16, 3))
ax_idx = 0

for (x_col, y_col, title) in lag_pairs:
    if x_col not in df_domain.columns or y_col not in df_domain.columns:
        continue
    joint = df_domain[[x_col, y_col]].dropna()
    if len(joint) < 20:
        continue
    lags = range(-4, 5)  # ±4 pasos de 6h = ±24h
    corrs = [joint[x_col].corr(joint[y_col].shift(-lag)) for lag in lags]
    best_lag = lags[np.argmax(np.abs(corrs))]
    lag_results.append({'Par': title, 'Mejor lag (pasos 6h)': best_lag,
                        'Mejor lag (h)': best_lag*6, 'Max |r|': round(max(np.abs(corrs)), 3)})

plt.close()

if lag_results:
    lag_df = pd.DataFrame(lag_results)
    display(lag_df)
    lag_df.to_csv(OUT_DIR / 'lag_analysis.csv', index=False)

    # Visualizar correlación cruzada
    fig, axes = plt.subplots(1, len(lag_pairs), figsize=(12, 4))
    if len(lag_pairs) == 1: axes = [axes]
    for ax, (x_col, y_col, title) in zip(axes, lag_pairs):
        if x_col not in df_domain.columns or y_col not in df_domain.columns:
            continue
        joint = df_domain[[x_col, y_col]].dropna()
        if len(joint) < 20:
            continue
        lags = range(-8, 9)
        corrs = [joint[x_col].corr(joint[y_col].shift(-lag)) for lag in lags]
        lag_hours = [l*6 for l in lags]
        ax.plot(lag_hours, corrs, marker='o', markersize=5, color='steelblue')
        ax.axhline(0, color='black', linewidth=0.5)
        ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('Lag (h)')
        ax.set_ylabel('Correlación Pearson')
    plt.suptitle('Análisis de lag — Efectos retardados', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'lag_analysis.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 7.5 Umbral ePAR para fotosíntesis activa ────────────────────────────────
# Referencia agronómica: ~20 µmol/m²/s = luz compensación; ~200 = fotosíntesis activa.

PAR_COMP    = 20    # µmol/m²/s — punto de compensación lumínica
PAR_ACTIVE  = 200   # µmol/m²/s — fotosíntesis activa baja
PAR_OPTIMAL = 600   # µmol/m²/s — rango óptimo aproximado para cultivos C3

epar_cols = [c for c in df_domain.columns if c.startswith('ePAR')]
if epar_cols:
    col = epar_cols[0]
    epar_data = df_domain[col].dropna()
    total = len(epar_data)
    below_comp   = (epar_data < PAR_COMP).sum()
    active_range = ((epar_data >= PAR_ACTIVE) & (epar_data < PAR_OPTIMAL)).sum()
    optimal      = (epar_data >= PAR_OPTIMAL).sum()
    
    print(f'Análisis de umbrales ePAR ({col}):')
    print(f'  < {PAR_COMP} µmol/m²/s (sin fotosíntesis neta): {below_comp} ({below_comp/total*100:.1f}%)')
    print(f'  {PAR_ACTIVE}–{PAR_OPTIMAL} µmol/m²/s (fotosíntesis activa-baja): {active_range} ({active_range/total*100:.1f}%)')
    print(f'  >= {PAR_OPTIMAL} µmol/m²/s (rango óptimo aprox.): {optimal} ({optimal/total*100:.1f}%)')
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(epar_data, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(PAR_COMP,    color='red',    linestyle='--', label=f'Compensación ({PAR_COMP})')
    ax.axvline(PAR_ACTIVE,  color='orange', linestyle='--', label=f'Fotosíntesis activa ({PAR_ACTIVE})')
    ax.axvline(PAR_OPTIMAL, color='green',  linestyle='--', label=f'Óptimo aprox. ({PAR_OPTIMAL})')
    ax.set_title(f'Distribución ePAR ({col}) con umbrales agronómicos', fontweight='bold')
    ax.set_xlabel('µmol/m²/s')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'epar_thresholds.png', dpi=120, bbox_inches='tight')
    plt.show()

### 7.1 Observaciones de dominio agrovoltaico

> **Evidencias clave para Sprint 2:**
>
> 1. **La posición STOW reduce drásticamente el ePAR** bajo el dosel. Durante el stow, el panel está a ~50° horizontal, bloqueando más luz. Este es el régimen más perjudicial para el cultivo.
> 2. **GPOA y ePAR no son linealmente proporcionales.** El ePAR bajo el dosel satura antes que el GPOA, lo que sugiere un efecto de sombreo no lineal de los paneles.
> 3. **S1 y S2 muestran diferencias sistemáticas** en VWC (~2–4 puntos porcentuales). Puede indicar diferencias en el tipo de suelo, profundidad de sensores, o estrategia de riego entre secciones.
> 4. **Efectos retardados:** la temperatura del suelo responde con ~6–12h de retardo respecto a la irradiancia. El VWC tiene efectos de retardo más largos (>24h). Relevante para el diseño del modelo.
> 5. **Umbrales ePAR:** existe un porcentaje significativo de timesteps en que el ePAR cae por debajo del umbral de compensación lumínica (~20 µmol/m²/s). Evaluar si coincide con posición de stow o con períodos nocturnos.

---
## 8. Hipótesis sobre sensores y espacios de investigación

A partir de los patrones encontrados en los datos, proponemos hipótesis razonadas sobre la estructura experimental de la instalación.

In [ ]:
# ── Análisis de clustering de sensores por comportamiento ──────────────────
# Usamos correlaciones entre trackers para inferir agrupaciones físicas.

df_track_num = dfs_num['tracking'].set_index('Time').sort_index()
tracker_cols = [c for c in df_track_num.columns]

if len(tracker_cols) >= 3:
    corr_trackers = df_track_num[tracker_cols].corr(method='spearman')
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_trackers, annot=True, fmt='.2f', cmap='RdYlGn',
                vmin=-1, vmax=1, center=0, linewidths=0.5,
                annot_kws={'size': 9}, ax=ax)
    ax.set_title('Correlaciones entre trackers (Spearman)\nInferencia de grupos físicos',
                 fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'tracker_correlations.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    # Identificar trackers "atascados" (baja varianza)
    print('\nVarianza de ángulo por tracker (baja varianza = posible stow fijo):')
    tracker_var = df_track_num[tracker_cols].var().sort_values()
    display(tracker_var.to_frame('Varianza (°²)'))

In [ ]:
# ── Comparativa de sensores ePAR para inferir zonas experimentales ──────────

epar_num = dfs_num['epar']
epar_cal_cols = [c for c in epar_num.columns if 'cal_out' in c and '16bit' not in c]

if epar_cal_cols:
    epar_ts = epar_num.set_index('Time')[epar_cal_cols].sort_index()
    
    # Mediana por sensor para un perfil promedio
    medians = epar_ts.median().sort_values(ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Perfil de medianas por sensor
    medians.plot(kind='bar', ax=axes[0], color='steelblue', width=0.7)
    axes[0].set_title('Mediana de ePAR por sensor\n(indicador de exposición lumínica)', fontweight='bold')
    axes[0].set_ylabel('µmol/m²/s')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Correlación entre sensores ePAR
    epar_corr = epar_ts.corr(method='spearman')
    sns.heatmap(epar_corr, annot=True, fmt='.2f', cmap='YlOrRd',
                linewidths=0.5, annot_kws={'size': 7}, ax=axes[1])
    axes[1].set_title('Correlación entre sensores ePAR\n(similitud de comportamiento lumínico)', fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'epar_sensor_profile.png', dpi=120, bbox_inches='tight')
    plt.show()

### 8.1 Hipótesis sobre espacios de investigación

Con base en la nomenclatura de los sensores y los patrones observados, proponemos las siguientes hipótesis:

| Identificador | Hipótesis | Evidencia base |
|:---|:---|:---|
| **R1** | Zona de referencia (control) sin cobertura de paneles | Nulos >95% en casi todos los sensores R1 durante el período analizado |
| **S1** | Sección experimental 1 con trackers M01–M05 | Sensores `S1d10`–`S1d21` con datos válidos; correlación con trackers M01–M05 |
| **S2** | Sección experimental 2 con trackers M06–M10 | Sensores `S2d30`–`S2d37` con datos válidos; diferencias sistemáticas vs. S1 |
| **M02, M06, M10** | Trackers con posible fallo mecánico o stow permanente | Varianza de ángulo muy baja, valor constante ≈50.6° |
| **WS100** | Estación meteorológica exterior de referencia | Prefijo `WS100`, valores de T aire ligeramente superiores (efecto isla de calor) |

> **Acción recomendada:** Validar con el equipo técnico de la planta la asignación de trackers a secciones y el estado de los trackers M02, M06 y M10.

---
## 9. Conclusiones del EDA y próximos pasos

Esta sección consolida los hallazgos del Sprint 1 en formato de informe ejecutivo.

In [ ]:
# ── Resumen ejecutivo de calidad de datos ───────────────────────────────────

print('='*70)
print('RESUMEN EJECUTIVO — EDA SPRINT 1')
print('='*70)

print('''
1. ESTADO DE LOS DATOS
   • 16 ficheros CSV identificados, de los cuales 8 son datasets de sensores
     a frecuencia baja/media (6h) y 5 son registros de precipitación de alta 
     frecuencia (~segundos, >600k filas).
   • Período con datos válidos: junio 2025 – presente (~9 meses de rango,
     pero con huecos significativos entre feb–jun 2025).
   • Zona R1: >95% nulos en todos los sensores. Probable zona de referencia
     no instrumentada o en fase de instalación durante el período analizado.
   • Unidades embebidas en strings (°C, W/m², %, km/h): limpieza necesaria
     antes de cualquier modelización.

2. PROBLEMAS DE CALIDAD DETECTADOS (riesgos para Sprint 2)
   ⚠ Trackers M02, M06, M10 con ángulo prácticamente constante (~50.6°):
     ¿fallo mecánico o posición de stow permanente? Requiere verificación.
   ⚠ Huecos temporales de varios días en algunos datasets.
   ⚠ Frecuencias heterogéneas (2h, 6h, ~segundos): resampleo temporal
     obligatorio antes de unir datasets para modelización.
   ⚠ Columnas 16bit en ePAR son duplicados de baja resolución; pueden omitirse.

3. VARIABLES MÁS RELEVANTES DEL PROYECTO
   ★ Energéticas:   track_M01–M09, GPOA_S1, GPOA_S2, Albedo_S1, Tpanel
   ★ Microclima:    ePAR (S1/S2), VWC (S1/S2), Tsoil (S1/S2), Tair_S1
   ★ Meteoro:       Tair_WS, wind_speed, wind_dir, precip_type

4. INSIGHTS CLAVE PARA SPRINT 2
   → La posición STOW (~50.6°) es el régimen más perjudicial para el cultivo
     (ePAR mínimo). Las reglas de rotación deben evitar el stow innecesario.
   → Existe un desfase temporal (lag ~6–12h) entre cambios de irradiancia y
     respuesta de temperatura del suelo. El modelo debe contemplar efectos 
     retardados.
   → S1 y S2 presentan diferencias sistemáticas en VWC y ePAR que sugieren
     distintas condiciones de base. Modelizar por sección o con variable
     indicadora de sección.
   → La relación tracking–ePAR es no lineal: ángulos intermedios pueden
     ser más favorables para el cultivo sin sacrificar toda la energía.
     Este es el corazón de la optimización multiobjetivo.

5. RIESGOS TÉCNICOS Y ANALÍTICOS
   ⚠ Período de datos válidos puede ser insuficiente para capturar variabilidad
     estacional completa (solo verano 2025).
   ⚠ Sin datos de la zona R1 no es posible establecer un control real.
   ⚠ Precipitación en alta frecuencia requiere agregación cuidadosa para
     no perder información sobre eventos puntuales relevantes.
   ⚠ Los trackers defectuosos contaminan el análisis de correlación;
     excluir o tratar aparte M02, M06, M10.

6. SIGUIENTES PASOS RECOMENDADOS (Sprint 2)
   □ Confirmar con equipo técnico: estado trackers M02/M06/M10, zona R1,
     asignación secciones S1/S2.
   □ Construir pipeline de limpieza reproducible: strip_unit + resampleo 6h
     + tratamiento de nulos por interpolación temporal.
   □ Diseñar variable objetivo: índice combinado Energía–Cultivo (GPOA + ePAR
     ponderados) para optimización multiobjetivo.
   □ Feature engineering: añadir hora del día, ángulo solar teórico,
     lag features de VWC y Tsoil, diferencias S1–S2.
   □ Primer modelo de reglas de rotación basado en árboles de decisión o
     regresión regularizada para interpretabilidad directa.
''')

In [ ]:
# ── Figura resumen de variables clave (exportable para presentación) ─────────

if not df_domain.empty and len(df_domain.dropna(thresh=5)) > 5:
    df_plot_final = df_domain.copy().sort_index()
    
    plot_vars = [
        ('track_M01', 'Ángulo Tracking M01 (°)', '#2c3e50'),
        ('GPOA_S1',   'GPOA S1 (W/m²)',          '#f39c12'),
        ('ePAR_S1d19','ePAR S1 (µmol/m²/s)',      '#27ae60'),
        ('VWC_S1d13', 'VWC S1 (%)',               '#2980b9'),
    ]
    valid_pv = [(col, lbl, col_) for col, lbl, col_ in plot_vars
                if col in df_plot_final.columns]
    
    if valid_pv:
        fig, axes = plt.subplots(len(valid_pv), 1, figsize=(14, 3*len(valid_pv)), sharex=True)
        if len(valid_pv) == 1: axes = [axes]
        fig.suptitle('Variables clave del sistema agrovoltaico — visión integrada',
                     fontweight='bold', fontsize=13)
        
        for ax, (col, lbl, color) in zip(axes, valid_pv):
            s = df_plot_final[col].dropna()
            ax.plot(s.index, s.values, color=color, linewidth=1, alpha=0.85)
            ax.set_ylabel(lbl, fontsize=9)
            ax.fill_between(s.index, s.values, alpha=0.1, color=color)
        
        axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m/%y'))
        axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(OUT_DIR / 'key_variables_overview.png', dpi=130, bbox_inches='tight')
        plt.show()
        print('=> Guardado en outputs/key_variables_overview.png')

In [ ]:
# ── Listado final de outputs generados ──────────────────────────────────────

print('=== Outputs generados en sprint1/outputs/ ===')
for f in sorted(OUT_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f'  {f.name:45s} {size:6.1f} KB')

---
## Apéndice: Supuestos explícitos y limitaciones del EDA

En línea con los principios del proyecto, se declaran explícitamente los supuestos y limitaciones de este análisis:

| Supuesto / Limitación | Impacto | Recomendación |
|:---|:---|:---|
| Zona R1 considerada como referencia no disponible | No hay grupo de control real en el análisis | Confirmar estado R1 con equipo técnico |
| Trackers M02, M06, M10 excluidos del análisis de correlación | Posible sesgo en patrones de tracking | Verificar mecánicamente |
| Período analizado: solo verano 2025 | Falta variabilidad estacional (invierno, otoño) | Ampliar periodo si se dispone de datos |
| Resampleo a 6h puede perder eventos breves de irradiancia | Infraestimación de picos de estrés | Revisar con resolución 2h para irradiancia |
| Umbrales ePAR basados en valores de referencia genéricos | Puede no representar el cultivo específico | Validar umbrales con especialista agrícola |
| No se dispone de información sobre el tipo de cultivo | No se pueden calcular índices agronómicos específicos | Solicitar metadatos al cliente |

---

*Notebook generado como Entregable E02 del Sprint 1 — Análisis y Optimización Operativa de Sistemas Agrovoltaicos Dinámicos.*  
*Equipo: Daniel Álvarez / Chacorocalfarobar | Organización: Sostenibilidad y Ciencia*